# Cochrane example: CMS bottom-up and improved-Hommel procedures

This notebook applies the current Python implementations to the 248 Cochrane studies, each containing five p-values. It uses `cms_thresholds_K5.json` as its only threshold file and does not require the legacy `.npy` files, R, or `rpy2`.

The comparison includes:

- BU $\Pi_{\mathrm{mix}}(0.3)$
- BU $\Pi_{\mathrm{mix}}(0.7)$
- BU $\Pi_1(0.3)$
- standard Hommel
- Gou
- improved Hommel with $\Pi_{\mathrm{mix}}$ and $\Pi_1$ at target powers 0.3 and 0.7


## 1. Imports and configuration

Run the notebook from the `BUStrongControl` directory. Only Python packages are used; R and `rpy2` are not required.

In [1]:
from pathlib import Path
import importlib
import json

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

sim_mod = importlib.import_module("simulate_cms_normal")
importlib.reload(sim_mod)
ih_mod = importlib.import_module("improved_hommel")
importlib.reload(ih_mod)

PROJECT_DIR = Path.cwd()
THRESHOLD_FILE = PROJECT_DIR / "cms_thresholds_K5.json"
PVALUE_FILE = PROJECT_DIR / "pvmat.csv"
COUNTS_OUTPUT = PROJECT_DIR / "cochrane_rejection_counts.csv"
DECISIONS_OUTPUT = PROJECT_DIR / "cochrane_rejection_decisions.csv"
TABLE1_OUTPUT = PROJECT_DIR / "cochrane_table1_bu_mix_vs_gou.csv"
TABLE2_OUTPUT = PROJECT_DIR / "cochrane_table2_discovery_summary.csv"
ALPHA = 0.05

BASE_METHODS = {
    r"BU Pi_mix(0.3)": ("BU", "BU_bayes_tp_0p3"),
    r"BU Pi_mix(0.7)": ("BU", "BU_bayes_tp_0p7"),
    r"BU Pi_1(0.3)": ("BU", "BU_pi1_tp_0p3"),
    "Hommel": ("comparison", "Hommel"),
    "Gou": ("comparison", "GTXR"),
}
IMPROVED_HOMMEL_METHODS = {
    r"Improved Hommel Pi_mix(0.3)": (
        "improved_hommel", "Improved_Hommel_bayes_tp_0p3"
    ),
    r"Improved Hommel Pi_mix(0.7)": (
        "improved_hommel", "Improved_Hommel_bayes_tp_0p7"
    ),
    r"Improved Hommel Pi_1(0.3)": (
        "improved_hommel", "Improved_Hommel_pi1_tp_0p3"
    ),
    r"Improved Hommel Pi_1(0.7)": (
        "improved_hommel", "Improved_Hommel_pi1_tp_0p7"
    ),
}
METHODS = {**BASE_METHODS, **IMPROVED_HOMMEL_METHODS}

for required_file in (THRESHOLD_FILE, PVALUE_FILE):
    if not required_file.exists():
        raise FileNotFoundError(f"Required file not found: {required_file}")

print("BU and improved-Hommel thresholds:", THRESHOLD_FILE)
print("P-values:", PVALUE_FILE)

BU and improved-Hommel thresholds: /Users/rajeshkarmakar/Downloads/BUStrongControl/cms_thresholds_K5.json
P-values: /Users/rajeshkarmakar/Downloads/BUStrongControl/pvmat.csv


## 2. Load the single JSON threshold file

For each normal-alternative procedure,

$$
\theta_q=\Phi^{-1}(\alpha/K)-\Phi^{-1}(q),
$$

where $q$ is the target Bonferroni power.

`cms_thresholds_K5.json` contains:

- $t_2,\ldots,t_5$ for all configured BU procedures;
- objective-specific last-step thresholds for improved Hommel.

The notebook does not load `cochrane_thresholds_K5.json` or `improved_hommel_thresholds_K5.json`.


In [2]:
with THRESHOLD_FILE.open("r", encoding="utf-8") as f:
    threshold_payload = json.load(f)

params = threshold_payload["params"]
K = int(params["K"])
if K != 5:
    raise ValueError(f"This Cochrane dataset requires K=5 thresholds; found K={K}.")
if not np.isclose(float(params["alpha"]), ALPHA):
    raise ValueError("BU threshold alpha does not match the notebook alpha.")

if "improved_hommel" not in threshold_payload:
    raise ValueError(
        "cms_thresholds_K5.json is missing its improved_hommel section."
    )
improved_hommel_payload = threshold_payload["improved_hommel"]
improved_params = improved_hommel_payload["params"]
if int(improved_params["K"]) != K:
    raise ValueError("Improved-Hommel threshold dimension does not match K.")
if not np.isclose(float(improved_params["alpha"]), ALPHA):
    raise ValueError("Improved-Hommel alpha does not match the notebook alpha.")

improved_hommel_by_name = {
    specification["name"]: specification
    for specification in improved_hommel_payload["procedures"]
}
missing_improved = [
    key
    for method_type, key in IMPROVED_HOMMEL_METHODS.values()
    if key not in improved_hommel_by_name
]
if missing_improved:
    raise ValueError(
        "Missing improved-Hommel threshold specifications: "
        + ", ".join(missing_improved)
    )

bu_threshold_rows = []
for spec in threshold_payload["bu_procedures"]:
    row = {
        "name": spec["name"],
        "objective": spec["objective"],
        "target_power": spec["target_power"],
        "theta": spec["theta"],
    }
    row.update({
        f"t{k}": value
        for k, value in enumerate(spec["thresholds_t2_to_tK"], start=2)
    })
    bu_threshold_rows.append(row)

print("BU thresholds")
display(pd.DataFrame(bu_threshold_rows))
print("Improved-Hommel last-step thresholds")
display(pd.DataFrame(improved_hommel_payload["procedures"]))


BU thresholds


,name,objective,target_power,theta,t2,t3,t4,t5
0,BU_bayes_tp_0p3,bayes,0.30,-1.801947,9.357506,15.022305,20.629294,25.327372
1,BU_bayes_tp_0p7,bayes,0.70,-2.850748,5.075213,9.083286,13.180291,17.495837
2,BU_bayes_tp_0p95,bayes,0.95,-3.971202,0.899656,1.797525,2.897422,4.125555
3,BU_pi1_tp_0p3,pi1,0.30,-1.801947,6.708739,9.043679,11.054412,12.867444
4,BU_pi1_tp_0p7,pi1,0.70,-2.850748,4.546185,7.304803,10.038490,12.764885
5,BU_pi1_tp_0p95,pi1,0.95,-3.971202,0.885361,1.725101,2.686309,3.754200


Improved-Hommel last-step thresholds


,name,objective,target_power,theta,threshold,empirical_tail_probability
0,Improved_Hommel_bayes_tp_0p3,bayes,0.30,-1.801947,1.038136,0.049999
1,Improved_Hommel_bayes_tp_0p7,bayes,0.70,-2.850748,0.566716,0.049999
2,Improved_Hommel_bayes_tp_0p95,bayes,0.95,-3.971202,0.129102,0.049999
3,Improved_Hommel_pi1_tp_0p3,pi1,0.30,-1.801947,12.886627,0.049999
4,Improved_Hommel_pi1_tp_0p7,pi1,0.70,-2.850748,12.795005,0.049999
5,Improved_Hommel_pi1_tp_0p95,pi1,0.95,-3.971202,3.766532,0.049999


## 3. Load and validate the Cochrane p-values

`pvmat.csv` has one row per study and five p-values per row. The first CSV column is the stored study index.

In [3]:
pvalue_table = pd.read_csv(PVALUE_FILE, index_col=0)
pvalue_table = pvalue_table.apply(pd.to_numeric, errors="raise")
pvalue_table.index.name = "study"

if pvalue_table.shape[1] != K:
    raise ValueError(
        f"Expected {K} p-values per study; found {pvalue_table.shape[1]}."
    )
if pvalue_table.isna().any().any():
    raise ValueError("The p-value table contains missing values.")
if ((pvalue_table < 0.0) | (pvalue_table > 1.0)).any().any():
    raise ValueError("All p-values must lie in [0, 1].")

print(f"Loaded {len(pvalue_table)} studies with {K} p-values each.")
display(pvalue_table.head())

Loaded 248 studies with 5 p-values each.


,V1,V2,V3,V4,V5
study,,,,,
1,0.040845,0.000049,0.000010,1.077239e-06,0.000244
2,0.000000,0.001735,0.000009,1.488535e-09,0.000000
3,0.257898,0.345134,0.281113,7.946676e-01,0.332110
4,0.819227,0.113165,0.147548,1.475484e-01,0.113165
5,0.026953,0.253826,0.044850,6.441857e-01,0.187666


## 4. Apply the current procedures

The analysis uses:

- `cms_bottomup.py` for the BU testing algorithm;
- `improved_hommel.py` for the improved-Hommel procedure;
- the optimized native Python standard-Hommel implementation;
- the native Python Gou/GTXR implementation;
- `cms_thresholds_K5.json` as the only source of calibrated thresholds.


In [4]:
def apply_selected_methods(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    base_result = sim_mod.apply_all_procedures_to_pvalues(
        pvalues,
        str(THRESHOLD_FILE),
    )

    selected = {}
    for label, (method_type, key) in METHODS.items():
        if method_type == "BU":
            decisions = base_result["BU"][key]["decisions"]
        elif method_type == "comparison":
            decisions = base_result[key]
        elif method_type == "improved_hommel":
            specification = improved_hommel_by_name[key]
            improved_result = ih_mod.apply_improved_hommel(
                p_values=pvalues,
                theta=float(specification["theta"]),
                threshold=float(specification["threshold"]),
                objective=specification["objective"],
                alpha=ALPHA,
            )
            decisions = improved_result["decisions"]
        else:
            raise ValueError(f"Unknown method type: {method_type}")
        selected[label] = np.asarray(decisions, dtype=int)
    return selected


def decisions_for_study(study_index):
    pvalues = pvalue_table.loc[study_index].to_numpy(dtype=float)
    selected = apply_selected_methods(pvalues)
    table = pd.DataFrame({"p_value": pvalues}, index=pvalue_table.columns)
    for method, decisions in selected.items():
        table[method] = decisions
    table.index.name = "hypothesis"
    return table

## 5. Analyze all 248 studies and export CSV files

Two outputs are generated:

- `cochrane_rejection_counts.csv`: number of rejected hypotheses by study and method;
- `cochrane_rejection_decisions.csv`: 0/1 decision for every study, method, and hypothesis.

The tables include the five-method base comparison and four improved-Hommel variants.


In [5]:
decisions_by_method = {method: [] for method in METHODS}
count_records = []

for study_index, row in pvalue_table.iterrows():
    selected = apply_selected_methods(row.to_numpy(dtype=float))
    count_record = {"study": study_index}
    for method, decisions in selected.items():
        decisions_by_method[method].append(decisions)
        count_record[method] = int(decisions.sum())
    count_records.append(count_record)

rejection_counts = pd.DataFrame(count_records).set_index("study")
rejection_counts.index.name = "study"

rejection_decisions = pd.concat(
    {
        method: pd.DataFrame(
            np.vstack(method_decisions),
            index=pvalue_table.index,
            columns=pvalue_table.columns,
        )
        for method, method_decisions in decisions_by_method.items()
    },
    axis=1,
)
rejection_decisions.index.name = "study"
rejection_decisions.columns.names = ["method", "hypothesis"]

rejection_counts.to_csv(COUNTS_OUTPUT)
rejection_decisions.to_csv(DECISIONS_OUTPUT)

print("Saved:", COUNTS_OUTPUT)
print("Saved:", DECISIONS_OUTPUT)
display(rejection_counts.head())

Saved: /Users/rajeshkarmakar/Downloads/BUStrongControl/cochrane_rejection_counts.csv
Saved: /Users/rajeshkarmakar/Downloads/BUStrongControl/cochrane_rejection_decisions.csv


,BU Pi_mix(0.3),BU Pi_mix(0.7),BU Pi_1(0.3),Hommel,Gou,Improved Hommel Pi_mix(0.3),Improved Hommel Pi_mix(0.7),Improved Hommel Pi_1(0.3),Improved Hommel Pi_1(0.7)
study,,,,,,,,,
1,5,5,5,5,5,5,5,5,5
2,5,5,5,5,5,5,5,5,5
3,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0


## 6. Overall rejection summary

In [6]:
rejection_summary = pd.DataFrame({
    "studies_with_any_rejection": (rejection_counts > 0).sum(),
    "total_rejections": rejection_counts.sum(),
    "mean_rejections_per_study": rejection_counts.mean(),
    "maximum_rejections_in_a_study": rejection_counts.max(),
})
rejection_summary.index.name = "method"
display(rejection_summary)

,studies_with_any_rejection,total_rejections,mean_rejections_per_study,maximum_rejections_in_a_study
method,,,,
BU Pi_mix(0.3),166,433,1.745968,5
BU Pi_mix(0.7),165,431,1.737903,5
BU Pi_1(0.3),158,414,1.669355,5
Hommel,157,414,1.669355,5
Gou,160,417,1.681452,5
Improved Hommel Pi_mix(0.3),162,419,1.689516,5
Improved Hommel Pi_mix(0.7),162,419,1.689516,5
Improved Hommel Pi_1(0.3),158,415,1.673387,5
Improved Hommel Pi_1(0.7),158,415,1.673387,5


## 7. BU $\Pi_{\mathrm{mix}}(0.3)$ versus Gou cross-tabulation

Cross-tabulate the number of discoveries from BU $\Pi_{\mathrm{mix}}(0.3)$ and Gou. The values are calculated from the current general thresholds and are not required to match previously published tables.


In [7]:
bu_mix_counts = rejection_counts[r"BU Pi_mix(0.3)"]
gou_counts = rejection_counts["Gou"]

table1 = pd.crosstab(bu_mix_counts, gou_counts).reindex(
    index=range(pvalue_table.shape[1] + 1),
    columns=range(pvalue_table.shape[1] + 1),
    fill_value=0,
)
table1.index.name = r"BU Pi_mix"
table1.columns.name = "Gou et al. (2014)"

bu_more = int((bu_mix_counts > gou_counts).sum())
gou_more = int((gou_counts > bu_mix_counts).sum())
bu_only = int(((bu_mix_counts > 0) & (gou_counts == 0)).sum())
gou_only = int(((gou_counts > 0) & (bu_mix_counts == 0)).sum())

table1.to_csv(TABLE1_OUTPUT)
display(table1)
print(f"BU Pi_mix has more discoveries in {bu_more} analyses.")
print(f"Gou has more discoveries in {gou_more} analyses.")
print(f"BU-only discoveries occur in {bu_only} analyses; Gou-only in {gou_only}.")
print("Saved:", TABLE1_OUTPUT)


Gou et al. (2014),0,1,2,3,4,5
BU Pi_mix,,,,,,
0,82,0,0,0,0,0
1,4,44,1,0,0,0
2,2,5,32,1,0,0
3,0,0,3,26,1,0
4,0,0,1,1,19,0
5,0,0,0,0,0,26


BU Pi_mix has more discoveries in 16 analyses.
Gou has more discoveries in 3 analyses.
BU-only discoveries occur in 6 analyses; Gou-only in 0.
Saved: /Users/rajeshkarmakar/Downloads/BUStrongControl/cochrane_table1_bu_mix_vs_gou.csv


## 8. Discovery summary

Summarize the average number of discoveries and the fraction of analyses with at least one discovery for the selected procedures. The values are calculated from the current thresholds in `cms_thresholds_K5.json`.


In [8]:
table2_methods = {
    r"BU Pi_mix": r"BU Pi_mix(0.3)",
    r"BU Pi_1": r"BU Pi_1(0.3)",
    r"IH Pi_mix": r"Improved Hommel Pi_mix(0.3)",
    r"IH Pi_1": r"Improved Hommel Pi_1(0.3)",
    "Hommel": "Hommel",
    "Gou": "Gou",
}

table2 = pd.DataFrame(
    {
        label: [
            rejection_counts[source].mean(),
            (rejection_counts[source] > 0).mean(),
        ]
        for label, source in table2_methods.items()
    },
    index=[
        "Average number of discoveries",
        "Fraction of at least one discovery",
    ],
)
table2.index.name = "Method"

table2.to_csv(TABLE2_OUTPUT, float_format="%.6f")
display(table2.round(3))
print("Saved:", TABLE2_OUTPUT)


,BU Pi_mix,BU Pi_1,IH Pi_mix,IH Pi_1,Hommel,Gou
Method,,,,,,
Average number of discoveries,1.746,1.669,1.690,1.673,1.669,1.681
Fraction of at least one discovery,0.669,0.637,0.653,0.637,0.633,0.645


Saved: /Users/rajeshkarmakar/Downloads/BUStrongControl/cochrane_table2_discovery_summary.csv


## 9. Inspect an individual study

Change `STUDY_INDEX` to inspect another row. The table reports each original p-value and the corresponding 0/1 decisions.

In [9]:
STUDY_ROW = 24  # Zero-based row position, matching the legacy notebook.
study_index = pvalue_table.index[STUDY_ROW]
study_table = decisions_for_study(study_index)
print(f"Row position {STUDY_ROW}; stored study index {study_index}")
display(study_table)

Row position 24; stored study index 25


,p_value,BU Pi_mix(0.3),BU Pi_mix(0.7),BU Pi_1(0.3),Hommel,Gou,Improved Hommel Pi_mix(0.3),Improved Hommel Pi_mix(0.7),Improved Hommel Pi_1(0.3),Improved Hommel Pi_1(0.7)
hypothesis,,,,,,,,,,
V1,0.021833,1,0,0,0,0,0,0,0,0
V2,0.176277,0,0,0,0,0,0,0,0,0
V3,0.313892,0,0,0,0,0,0,0,0,0
V4,0.011126,1,1,0,0,1,1,1,0,0
V5,0.196924,0,0,0,0,0,0,0,0,0


## 10. Find studies where methods disagree

This reproduces the purpose of the legacy “interesting index” list without hard-coding study numbers.

In [10]:
disagreement_mask = rejection_counts.nunique(axis=1) > 1
disagreement_counts = rejection_counts.loc[disagreement_mask]

print(
    f"Methods disagree on the number of rejections in "
    f"{len(disagreement_counts)} of {len(rejection_counts)} studies."
)
display(disagreement_counts)

Methods disagree on the number of rejections in 23 of 248 studies.


method,BU Pi_mix(0.3),BU Pi_mix(0.7),BU Pi_1(0.3),Hommel,Gou,Improved Hommel Pi_mix(0.3),Improved Hommel Pi_mix(0.7),Improved Hommel Pi_1(0.3),Improved Hommel Pi_1(0.7)
study,,,,,,,,,
10,1,1,0,0,0,0,0,0,0
21,1,1,0,0,1,1,1,0,0
22,1,1,0,0,1,1,1,0,0
25,2,1,0,0,1,1,1,0,0
33,2,2,0,0,0,0,0,0,0
35,4,4,3,3,3,3,3,3,3
46,4,4,2,2,2,2,2,2,2
55,3,2,1,2,2,2,2,2,2
85,2,3,3,3,3,3,3,3,3


## Output interpretation

A value of 1 in `cochrane_rejection_decisions.csv` means that the corresponding procedure rejects that hypothesis at family-wise level $\alpha=0.05$. The count table is obtained by summing those five decisions within each study.

“Improved Hommel” refers to the objective-specific last-step statistical enhancement, not merely the faster implementation of standard Hommel.

The notebook performs a real-data multiple-testing comparison; it does not estimate power or FWER because the true null/non-null status of each Cochrane hypothesis is unknown.
